# 1 — CLIP loops over random image sources

This experiment repeatedly fetches an image, embeds it with CLIP, and compares it with either:

- a **fixed text target**, or
- a **fixed image target**.

The loop tracks the current, best, and worst samples. In the Gradio UI it runs until the event is cancelled. In command-line scripts it is finite by default so jobs terminate predictably.

In [ ]:
# Uncomment once in a fresh environment.
# %pip install -e "..[ui,notebooks]"

## Sources and network behavior

The repository includes two source aliases:

- `picsum` → random stock-like photographs.
- `thispersondoesnotexist` → generated faces.

Every request gets a cache-busting query parameter. Network errors are expected in a long-running stream; production code should add retries, backoff, and source-specific rate limits.

In [ ]:
from clip_token_lab.image_sources import SOURCES, fetch_random_image

SOURCES

## Text → random images

The target text is embedded once. Each fetched image then requires only an image forward pass and one dot product.

In [ ]:
from clip_token_lab.clip import ClipEmbedder
from clip_token_lab.loops import iter_text_to_random_images

clip = ClipEmbedder()

# A finite notebook example. Increase iterations deliberately.
# for state in iter_text_to_random_images(
#     "a smiling person", clip, source="picsum", iterations=5, delay=0.25
# ):
#     print(state.iteration, state.current_score, state.best_score, state.worst_score)

## Image → random images

The target image is also embedded once. The remaining loop is identical, so both UI pathways share the same internal iterator.

In [ ]:
from clip_token_lab.io import load_rgb_image
from clip_token_lab.loops import iter_image_to_random_images

# target = load_rgb_image("target.jpg")
# for state in iter_image_to_random_images(
#     target, clip, source="picsum", iterations=5, delay=0.25
# ):
#     print(state.iteration, state.current_score, state.best_score, state.worst_score)

## Why the UI uses a generator

A Gradio callback can yield intermediate values. The start event holds a reference to that generator, and the stop button cancels the event. The core loop itself does not import Gradio, which keeps it usable from tests and scripts.

In [ ]:
from clip_token_lab.apps.clip_loops import build_demo

demo = build_demo()
demo.launch(inline=True)

## Minimal scripts

- `scripts/loops/text_to_random_images.py`
- `scripts/loops/image_to_random_images.py`

Both accept `--iterations`, `--delay`, and `--source`, then save the final best and worst images.